# Part 8: Sharding and Multi-Device

**Goal**: Learn JAX's modern [`jax.sharding`](https://jax.readthedocs.io/en/latest/jax.sharding.html) API — how to distribute data and computation across devices using meshes, partition specs, and sharding constraints. By the end, you'll shard the decoder from Notebook 07 for multi-device inference.

---

## Table of Contents

1. **The Multi-Device Mental Model** — Arrays on devices
2. **Meshes — Naming Your Device Topology** — Logical grids over physical hardware
3. **NamedSharding and PartitionSpec** — Declaring how data is distributed
4. **jit with Sharding Constraints** — Controlling compilation
5. **Data Parallelism** — Replicate params, shard batches
6. **Model (Tensor) Parallelism** — Shard weight matrices
7. **Debugging Sharding** — Visualization and common mistakes
8. **Common Misconceptions** — What experienced engineers get wrong
9. **Capstone: Sharded Decoder Inference** — Multi-device generation
10. **Summary — What To Do Next**

---

> **Prerequisites**: This notebook builds on **Notebook 07: Control Flow**. We'll reuse the autoregressive decoder and KV-cache. The setup cell below reconstructs the essentials.

> **Note on devices**: This notebook works on any hardware. With a single device, sharding operations still work — JAX simulates the requested distribution. With multiple hardware accelerators, the same code runs with real distribution. We use [`jax.devices()`](https://jax.readthedocs.io/en/latest/_autosummary/jax.devices.html) to discover what's available.


In [1]:
# @title Setup { display-mode: "form" }

import jax
import jax.numpy as jnp
import jax.lax as lax
import numpy as np
import matplotlib.pyplot as plt
import time
from jax.sharding import Mesh, NamedSharding, PartitionSpec as P

plt.style.use('default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

devices = jax.devices()
n_devices = len(devices)
print(f"JAX version: {jax.__version__}")
print(f"Backend:     {jax.default_backend()}")
print(f"Devices:     {n_devices} × {devices[0].platform.upper()}")
for i, d in enumerate(devices):
    print(f"  [{i}] {d}")

JAX version: 0.9.2
Backend:     cpu
Devices:     1 × CPU
  [0] TFRT_CPU_0


In [2]:
# @title Reconstruct Decoder from Notebook 07 { display-mode: "form" }

# --- Configuration ---
VOCAB_SIZE = 64
D_MODEL = 64
N_HEADS = 4
D_HEAD = D_MODEL // N_HEADS
D_FF = 128
N_LAYERS = 2
MAX_SEQ_LEN = 128

# --- Initialization ---
def init_attention_params(key, d_model, n_heads, d_head):
    keys = jax.random.split(key, 4)
    scale = jnp.sqrt(2.0 / d_model)
    return {
        'wq': jax.random.normal(keys[0], (d_model, n_heads, d_head)) * scale,
        'wk': jax.random.normal(keys[1], (d_model, n_heads, d_head)) * scale,
        'wv': jax.random.normal(keys[2], (d_model, n_heads, d_head)) * scale,
        'wo': jax.random.normal(keys[3], (n_heads, d_head, d_model)) * scale,
    }

def init_ff_params(key, d_model, d_ff):
    k1, k2 = jax.random.split(key)
    return {
        'w1': jax.random.normal(k1, (d_model, d_ff)) * jnp.sqrt(2.0 / d_model),
        'b1': jnp.zeros(d_ff),
        'w2': jax.random.normal(k2, (d_ff, d_model)) * jnp.sqrt(2.0 / d_ff),
        'b2': jnp.zeros(d_model),
    }

def init_decoder_params(key):
    keys = jax.random.split(key, 2 + 2 * N_LAYERS)
    params = {
        'embedding': jax.random.normal(keys[0], (VOCAB_SIZE, D_MODEL)) * 0.02,
        'unembed': jax.random.normal(keys[1], (D_MODEL, VOCAB_SIZE)) * 0.02,
        'layers': [],
    }
    for i in range(N_LAYERS):
        params['layers'].append({
            'attn': init_attention_params(keys[2 + 2*i], D_MODEL, N_HEADS, D_HEAD),
            'ff': init_ff_params(keys[2 + 2*i + 1], D_MODEL, D_FF),
        })
    return params

# --- Forward pass ---
def attention_with_cache(attn_params, x, cache_k, cache_v, step):
    q = jnp.einsum('d,dhk->hk', x, attn_params['wq'])
    k = jnp.einsum('d,dhk->hk', x, attn_params['wk'])
    v = jnp.einsum('d,dhk->hk', x, attn_params['wv'])
    new_cache_k = cache_k.at[step].set(k)
    new_cache_v = cache_v.at[step].set(v)
    scores = jnp.einsum('hk,shk->hs', q, new_cache_k) / jnp.sqrt(D_HEAD)
    mask = jnp.arange(MAX_SEQ_LEN) <= step
    scores = jnp.where(mask[None, :], scores, -1e9)
    weights = jax.nn.softmax(scores, axis=-1)
    attn_out = jnp.einsum('hs,shk->hk', weights, new_cache_v)
    output = jnp.einsum('hk,hkd->d', attn_out, attn_params['wo'])
    return output, new_cache_k, new_cache_v

def feedforward(ff_params, x):
    h = jnp.maximum(x @ ff_params['w1'] + ff_params['b1'], 0)
    return h @ ff_params['w2'] + ff_params['b2']

def decoder_step(params, token_id, cache, step):
    x = params['embedding'][token_id]
    new_cache = []
    for i, layer_params in enumerate(params['layers']):
        cache_k, cache_v = cache[i]
        attn_out, new_ck, new_cv = attention_with_cache(
            layer_params['attn'], x, cache_k, cache_v, step)
        x = x + attn_out
        ff_out = feedforward(layer_params['ff'], x)
        x = x + ff_out
        new_cache.append((new_ck, new_cv))
    logits = x @ params['unembed']
    return logits, new_cache

def make_empty_cache():
    return [(jnp.zeros((MAX_SEQ_LEN, N_HEADS, D_HEAD)),
             jnp.zeros((MAX_SEQ_LEN, N_HEADS, D_HEAD))) for _ in range(N_LAYERS)]

params = init_decoder_params(jax.random.PRNGKey(42))
n_params = sum(x.size for x in jax.tree.leaves(params))
print(f"Decoder reconstructed: {N_LAYERS} layers, {n_params:,} parameters")

Decoder reconstructed: 2 layers, 74,112 parameters


---

# 1. The Multi-Device Mental Model

In JAX, every array lives on one or more devices. There is no separate "distributed mode" you switch into — the same code runs on 1 device or 1,000 devices. The difference is how you **shard** (distribute) your arrays.

> **The key idea**: you declare how data should be distributed, and JAX + XLA figure out the communication (all-reduce, all-gather, etc.) automatically.

There are three distribution strategies:

| Strategy | Arrays | Use Case |
|---|---|---|
| **Replicated** | Full copy on every device | Small params, broadcast data |
| **Sharded** | Split across devices | Large weights, large batches |
| **Partially replicated** | Replicated within groups, sharded across | Hybrid parallelism |

In [3]:
# By default, arrays live on the first device
x = jnp.ones((4, 4))
print(f"Default array devices: {x.devices()}")
print(f"Number of devices: {len(x.devices())}")

# You can check if an array is on a single device or sharded
print(f"\nSharding info: {x.sharding}")

Default array devices: {CpuDevice(id=0)}
Number of devices: 1

Sharding info: SingleDeviceSharding(device=CpuDevice(id=0), memory_kind=device)


---

# 2. Meshes — Naming Your Device Topology

A **mesh** is a logical grid imposed on your physical devices. You name the axes, and then refer to those names when describing how to shard data.

> **Mental model**: think of a mesh as a coordinate system for your devices. If your mesh has a `'data'` axis of size 4, each device gets a coordinate along that axis (0, 1, 2, 3). A `PartitionSpec` then says "split this array axis across the `'data'` mesh axis" — meaning device 0 gets slice 0, device 1 gets slice 1, etc.

```
Physical:  [Device0, Device1, Device2, Device3]

1D Mesh (data parallelism):
  axis 'data': [Device0, Device1, Device2, Device3]

2D Mesh (data + model parallelism):
  axis 'data':  [Device0, Device1]  [GPU2, Device3]
  axis 'model': [Device0, Device2]  [GPU1, Device3]
```

The mesh doesn't move data — it just assigns names to device arrangements.

<br>

```mermaid
graph TD
    Global["Global Batch Array: f32[128, 512]"] -->|PartitionSpec| Mesh
    
    subgraph Device Mesh Topology
    D0((TPU 0)) -.- D1((TPU 1))
    D2((TPU 2)) -.- D3((TPU 3))
    D0 -.- D2
    D1 -.- D3
    end
```


In [4]:
# 1D mesh: all devices along one axis named 'data'
mesh_1d = Mesh(np.array(devices), axis_names=('data',))
print(f"1D Mesh: {mesh_1d.shape}")
print(f"  Axis 'data': {mesh_1d.shape['data']} devices")

# 2D mesh (if we have >= 4 devices, otherwise demonstrate the concept)
if n_devices >= 4:
    device_grid = np.array(devices[:4]).reshape(2, 2)
    mesh_2d = Mesh(device_grid, axis_names=('data', 'model'))
    print(f"\n2D Mesh: {mesh_2d.shape}")
    print(f"  Axis 'data':  {mesh_2d.shape['data']} devices")
    print(f"  Axis 'model': {mesh_2d.shape['model']} devices")
else:
    print(f"\n(Only {n_devices} device(s) — 2D mesh examples will use 1D mesh)")
    print(f"All sharding concepts apply; with 1 device, sharding is a no-op.")

# The mesh we'll use throughout this notebook
mesh = Mesh(np.array(devices), axis_names=('data',))
print(f"\nWorking mesh: {mesh.shape}")

1D Mesh: OrderedDict({'data': 1})
  Axis 'data': 1 devices

(Only 1 device(s) — 2D mesh examples will use 1D mesh)
All sharding concepts apply; with 1 device, sharding is a no-op.

Working mesh: OrderedDict({'data': 1})


---

# 3. NamedSharding and PartitionSpec

A `PartitionSpec` declares how each axis of an array maps to mesh axes:

```python
P('data', None)  # Shard axis 0 across 'data', replicate axis 1
P(None, 'model') # Replicate axis 0, shard axis 1 across 'model'
P()              # Fully replicated (no sharding)
P('data')        # 1D array sharded across 'data'
```

`NamedSharding` combines a mesh with a partition spec to create a concrete sharding.

> **`None` vs. a mesh axis name**: `None` in a `PartitionSpec` position means "replicate this array axis — every device gets a full copy." A mesh axis name means "split this array axis across those devices — each device gets a different slice." A fully replicated array (`P()`) is identical on every device; a fully sharded array (`P('data', 'model')`) has no device holding a complete copy.

In [5]:
# Create a sharding: split axis 0 across 'data' devices
sharding_split = NamedSharding(mesh, P('data'))

# Create a sharding: fully replicated
sharding_replicated = NamedSharding(mesh, P())

# Apply shardings with jax.device_put
x = jnp.arange(16).reshape(4, 4).astype(jnp.float32)

x_split = jax.device_put(x, sharding_split)
x_replicated = jax.device_put(x, sharding_replicated)

print("Original array:")
print(x)
print(f"\nSplit across 'data' axis (P('data')):")
print(f"  Devices: {x_split.devices()}")
print(f"  Sharding: {x_split.sharding}")

print(f"\nFully replicated (P()):")
print(f"  Devices: {x_replicated.devices()}")
print(f"  Sharding: {x_replicated.sharding}")

Original array:
[[ 0.  1.  2.  3.]
 [ 4.  5.  6.  7.]
 [ 8.  9. 10. 11.]
 [12. 13. 14. 15.]]

Split across 'data' axis (P('data')):
  Devices: {CpuDevice(id=0)}
  Sharding: NamedSharding(mesh=Mesh('data': 1, axis_types=(Auto,)), spec=P('data',), memory_kind=device)

Fully replicated (P()):
  Devices: {CpuDevice(id=0)}
  Sharding: NamedSharding(mesh=Mesh('data': 1, axis_types=(Auto,)), spec=P(), memory_kind=device)


In [6]:
# @title Visualizing Array Sharding { display-mode: "form" }

def visualize_sharding_simple(arr, title=""):
    """Simple visualization of how an array is distributed."""
    sharding = arr.sharding
    print(f"\n{title}")
    print(f"  Array shape: {arr.shape}")
    print(f"  Partition spec: {sharding.spec}")

    if hasattr(sharding, 'mesh'):
        print(f"  Mesh: {sharding.mesh.shape}")

    # Show which devices hold which data
    try:
        shard_shape = arr.sharding.shard_shape(arr.shape)
        print(f"  Shard shape (per device): {shard_shape}")
    except:
        print(f"  (Shard shape unavailable)")

# Demonstrate different shardings on a matrix
x = jnp.ones((8, 4))

shardings = {
    "Replicated P()": P(),
    "Split rows P('data', None)": P('data', None),
    "Split cols P(None, 'data')": P(None, 'data'),
}

for name, spec in shardings.items():
    sharded_x = jax.device_put(x, NamedSharding(mesh, spec))
    visualize_sharding_simple(sharded_x, title=name)


Replicated P()
  Array shape: (8, 4)
  Partition spec: P()
  Mesh: OrderedDict({'data': 1})
  Shard shape (per device): (8, 4)

Split rows P('data', None)
  Array shape: (8, 4)
  Partition spec: P('data', None)
  Mesh: OrderedDict({'data': 1})
  Shard shape (per device): (8, 4)

Split cols P(None, 'data')
  Array shape: (8, 4)
  Partition spec: P(None, 'data')
  Mesh: OrderedDict({'data': 1})
  Shard shape (per device): (8, 4)


In [7]:
# JAX provides a built-in visualizer
x = jnp.ones((8, 4))

print("=== Replicated (P()) ===")
replicated = jax.device_put(x, NamedSharding(mesh, P()))
jax.debug.visualize_array_sharding(replicated)

print("\n=== Sharded rows (P('data', None)) ===")
sharded = jax.device_put(x, NamedSharding(mesh, P('data', None)))
jax.debug.visualize_array_sharding(sharded)

print("\nThis visualization shows which device holds which part of the array.")
print("It's your best friend for debugging sharding issues.")

=== Replicated (P()) ===


            
            
            
            
            
   CPU 0    
            
            
            
            
            


=== Sharded rows (P('data', None)) ===


            
            
            
            
            
   CPU 0    
            
            
            
            
            


This visualization shows which device holds which part of the array.
It's your best friend for debugging sharding issues.


---

# 4. jit with Sharding Constraints

You can tell [`jit`](https://jax.readthedocs.io/en/latest/_autosummary/jax.jit.html) how inputs should be sharded and how outputs should be arranged. This lets XLA plan the computation and communication together.

In [8]:
# A sharded matrix multiply
def matmul(x, w):
    return x @ w

# Declare shardings: shard x rows across data, replicate weights
x = jnp.ones((8, 4))
w = jnp.ones((4, 4))

x_sharding = NamedSharding(mesh, P('data', None))  # Split batch dim
w_sharding = NamedSharding(mesh, P())               # Replicate weights
out_sharding = NamedSharding(mesh, P('data', None)) # Split output batch dim

sharded_matmul = jax.jit(
    matmul,
    in_shardings=(x_sharding, w_sharding),
    out_shardings=out_sharding,
)

x_sharded = jax.device_put(x, x_sharding)
w_replicated = jax.device_put(w, w_sharding)

result = sharded_matmul(x_sharded, w_replicated)
print(f"Input x:  {x.shape}, sharded rows across {n_devices} devices")
print(f"Weight w: {w.shape}, replicated")
print(f"Output:   {result.shape}, sharded rows across {n_devices} devices")
print(f"\nEach device computed its chunk of rows independently — no communication needed!")

Input x:  (8, 4), sharded rows across 1 devices
Weight w: (4, 4), replicated
Output:   (8, 4), sharded rows across 1 devices

Each device computed its chunk of rows independently — no communication needed!


## with_sharding_constraint

Sometimes you want to control sharding *inside* a function — for example, ensuring that intermediate activations are sharded correctly.

In [9]:
from jax.lax import with_sharding_constraint

def constrained_fn(x, w):
    # Ensure intermediate result is sharded along data axis
    h = x @ w
    h = with_sharding_constraint(h, NamedSharding(mesh, P('data', None)))
    return jax.nn.relu(h)

# JIT infers the full sharding strategy including the constraint
sharded_fn = jax.jit(constrained_fn)
result = sharded_fn(x_sharded, w_replicated)
print(f"Output sharding: {result.sharding.spec}")
print("The constraint guided XLA's sharding decisions for the intermediate value.")

Output sharding: P('data', None)
The constraint guided XLA's sharding decisions for the intermediate value.


---

# 5. Data Parallelism — The Simple Case

Data parallelism is the most common distribution strategy: **replicate the model on every device, shard the batch**.

Each device processes its slice of the batch, computes local gradients, and the gradients are all-reduced (averaged) before the parameter update. In JAX, this happens automatically when you shard the batch and replicate the params.

| Strategy | What's distributed | Best when |
|---|---|---|
| **Data parallelism** | Batch (each device gets different data) | Model fits on one device; want throughput |
| **Tensor parallelism** | Weight matrices (each device holds a shard) | Model is too large for one device |
| **Pipeline parallelism** | Layers (each device runs different layers) | Very deep models; latency less critical |

> **What `jax.device_put(array, sharding)` actually does**: it doesn't just annotate the array — it physically sends different slices to different devices right now. After `device_put`, each device holds `array.shape[batch_axis] // n_devices` rows. You can verify this with `jax.debug.visualize_array_sharding`.

In [10]:
def init_mlp_pytree(key, layer_sizes):
    params = []
    for i in range(len(layer_sizes) - 1):
        key, w_key, b_key = jax.random.split(key, 3)
        fan_in, fan_out = layer_sizes[i], layer_sizes[i + 1]
        params.append({
            'w': jax.random.normal(w_key, (fan_in, fan_out)) * jnp.sqrt(2.0 / fan_in),
            'b': jnp.zeros(fan_out),
        })
    return params

def mlp_forward_pytree(params, x):
    for i, layer in enumerate(params):
        x = x @ layer['w'] + layer['b']
        if i < len(params) - 1:
            x = jnp.maximum(x, 0)
    return x

# Replicate params, shard batch
replicated = NamedSharding(mesh, P())
batch_sharded = NamedSharding(mesh, P('data'))
batch_sharded_2d = NamedSharding(mesh, P('data', None))

mlp_params = init_mlp_pytree(jax.random.PRNGKey(0), [8, 64, 32, 1])

# Replicate params across all devices
mlp_params = jax.device_put(mlp_params, replicated)

# Create a sharded batch
x_batch = jnp.ones((n_devices * 32, 8))  # Batch divisible by n_devices
y_batch = jnp.zeros(n_devices * 32)
x_batch = jax.device_put(x_batch, batch_sharded_2d)
y_batch = jax.device_put(y_batch, batch_sharded)

def loss_fn(params, x, y):
    preds = jax.vmap(lambda xi: mlp_forward_pytree(params, xi))(x).squeeze()
    return jnp.mean((preds - y) ** 2)

@jax.jit
def dp_train_step(params, x, y, lr=0.01):
    loss, grads = jax.value_and_grad(loss_fn)(params, x, y)
    # Gradients are automatically all-reduced because params are replicated
    params = jax.tree.map(lambda p, g: p - lr * g, params, grads)
    return params, loss

# Run a training step
new_params, loss = dp_train_step(mlp_params, x_batch, y_batch)
print(f"Loss: {loss.item():.4f}")
print(f"Params still replicated: {new_params[0]['w'].sharding.spec}")
print(f"\nGradients were all-reduced automatically because params are replicated.")
print(f"No manual all-reduce needed — the sharding spec tells XLA what to do.")


Loss: 2.3239
Params still replicated: P()

Gradients were all-reduced automatically because params are replicated.
No manual all-reduce needed — the sharding spec tells XLA what to do.


---

# 6. Model (Tensor) Parallelism

When a model's weights are too large for a single device, you **shard the weights** across devices. This is tensor parallelism: each device holds a slice of the weight matrix and computes a partial result.

For a linear layer `y = x @ W`:
- Shard `W` along the output dimension: each device holds `W[:, chunk]` and computes `y[:, chunk]`
- Shard `W` along the input dimension: each device holds `W[chunk, :]` and computes a partial sum → needs an all-reduce

The first approach (column-parallel) is simpler because outputs can be concatenated without communication.

In [11]:
# Column-parallel: shard weight along output dimension
# Each device computes y_chunk = x @ W_chunk

def tensor_parallel_linear(x, w, b):
    """Linear layer where w is sharded along axis 1 (output dim)."""
    y = x @ w + b
    return y

# Create a weight matrix sharded along the output dimension
d_in, d_out = 64, 128
w = jax.random.normal(jax.random.PRNGKey(0), (d_in, d_out))
b = jnp.zeros(d_out)
x = jax.random.normal(jax.random.PRNGKey(1), (d_in,))

# Shard weight columns and bias across devices
# With 1 device: this is a no-op. With N devices: each gets d_out/N columns.
w_sharded = jax.device_put(w, NamedSharding(mesh, P(None, 'data')))
b_sharded = jax.device_put(b, NamedSharding(mesh, P('data')))
x_replicated = jax.device_put(x, NamedSharding(mesh, P()))

result = jax.jit(tensor_parallel_linear)(x_replicated, w_sharded, b_sharded)

print(f"Weight shape: {w.shape}, sharded P(None, 'data')")
print(f"  Per-device shard: ({d_in}, {d_out // n_devices})")
print(f"Input: {x.shape}, replicated")
print(f"Output: {result.shape}, sharding: {result.sharding.spec}")
print(f"\nEach device computed {d_out // n_devices} output features independently.")

Weight shape: (64, 128), sharded P(None, 'data')
  Per-device shard: (64, 128)
Input: (64,), replicated
Output: (128,), sharding: P()

Each device computed 128 output features independently.


---

# 7. Debugging Sharding

Sharding bugs are subtle — your code runs, produces results, but may be much slower than expected because of unnecessary communication (all-gathers, all-reduces). Here's how to catch them.

In [12]:
def check_sharding(name, arr):
    """Print sharding info for an array."""
    print(f"  {name:.<40s} shape={str(arr.shape):<15s} spec={arr.sharding.spec}")

print("Sharding audit:")
check_sharding("params[0]['w']", mlp_params[0]['w'])
check_sharding("x_batch", x_batch)
check_sharding("result", new_params[0]['w'])

print("\n✓ Params replicated (P()) — correct for data parallelism")
print("✓ Batch sharded (P('data', None)) — correct for data parallelism")
print("✓ Updated params still replicated — all-reduce happened correctly")

Sharding audit:
  params[0]['w'].......................... shape=(8, 64)         spec=P()
  x_batch................................. shape=(32, 8)         spec=P('data', None)
  result.................................. shape=(8, 64)         spec=P()

✓ Params replicated (P()) — correct for data parallelism
✓ Batch sharded (P('data', None)) — correct for data parallelism
✓ Updated params still replicated — all-reduce happened correctly


## Common Sharding Mistakes

| Symptom | Likely Cause | Fix |
|---|---|---|
| Unexpectedly slow | Unnecessary all-gather before matmul | Check intermediate sharding |
| OOM on one device | Unsharded large array | Apply appropriate PartitionSpec |
| Correct but slow | Batch not divisible by n_devices | Pad batch to multiple of n_devices |
| Wrong results | Sharding mismatch between function and data | Audit with [`jax.debug.visualize_array_sharding`](https://jax.readthedocs.io/en/latest/_autosummary/jax.debug.visualize_array_sharding.html) |

---

# 8. Common Misconceptions

## Misconception: "Sharding is only for massive models"

Even models that fit on a single device benefit from data parallelism. If you have 4 accelerators and a batch of 1024, sharding the batch across devices gives you ~4x throughput with minimal code changes. Sharding isn't just about fitting big models — it's about using all your hardware.


## Misconception: "I need pmap for multi-device computation"

[`pmap`](https://jax.readthedocs.io/en/latest/_autosummary/jax.pmap.html) was JAX's original multi-device API. It still works but is considered legacy. The modern `jax.sharding` API (meshes + partition specs) is:
- More flexible (supports 2D+ meshes, mixed strategies)
- More composable (works with [`jit`](https://jax.readthedocs.io/en/latest/_autosummary/jax.jit.html), not a separate transform)
- Better optimized by XLA (whole-program sharding analysis)

New code should use [`jax.sharding`](https://jax.readthedocs.io/en/latest/jax.sharding.html) exclusively.

## Misconception: "Replication is wasteful"

Replication is **intentional** in data parallelism. Every device needs the full model to compute gradients for its batch slice. The all-reduce of gradients is usually fast relative to computation. Replication becomes wasteful only when the model is too large to fit on one device — that's when you add tensor parallelism.

## Misconception: "Sharding is automatic — JAX figures it out"

JAX needs you to specify your **intent** — which arrays are sharded and how. Once you've declared the shardings, XLA figures out the communication patterns (all-reduce, all-gather, etc.). But if you don't specify shardings, everything stays on one device. The common pattern is: shard the inputs/outputs via `in_shardings`/`out_shardings` on [`jit`](https://jax.readthedocs.io/en/latest/_autosummary/jax.jit.html), and let XLA propagate shardings for intermediates.

---

# 9. Capstone: Sharded Decoder Inference

Let's take the decoder from Notebook 07 and run it across multiple devices. We'll implement:
1. **Replicated params** (the model fits on one device in our case)
2. **Sharded batch** for data-parallel inference
3. Batched autoregressive generation with sharded KV-cache

In [13]:
# @title Shard the Decoder Parameters { display-mode: "form" }

# Replicate all parameters (the model is small)
replicated_sharding = NamedSharding(mesh, P())
sharded_params = jax.device_put(params, replicated_sharding)

print("Parameter sharding:")
param_leaves = jax.tree.leaves(sharded_params)
for leaf in param_leaves[:4]:  # Show first few
    print(f"  shape={str(leaf.shape):<25s} spec={leaf.sharding.spec}")
print(f"  ... ({len(param_leaves)} total parameter arrays, all replicated)")

Parameter sharding:
  shape=(64, 64)                  spec=P()
  shape=(64, 4, 16)               spec=P()
  shape=(4, 16, 64)               spec=P()
  shape=(64, 4, 16)               spec=P()
  ... (18 total parameter arrays, all replicated)


In [14]:
# @title Batched Decoder Step { display-mode: "form" }

def batched_decoder_step(params, token_ids, caches, step):
    """Process a batch of tokens through the decoder.

    Args:
        params: replicated decoder params
        token_ids: (batch,) array of token IDs
        caches: list of (batch_cache_k, batch_cache_v) per layer
                each cache: (batch, max_seq_len, n_heads, d_head)
        step: current sequence position

    Returns:
        logits: (batch, vocab_size)
        new_caches: updated caches
    """
    batch_size = token_ids.shape[0]

    # Embed: (batch, d_model)
    x = params['embedding'][token_ids]

    new_caches = []
    for i, layer_params in enumerate(params['layers']):
        cache_k, cache_v = caches[i]

        # Project Q, K, V for entire batch
        q = jnp.einsum('bd,dhk->bhk', x, layer_params['attn']['wq'])
        k = jnp.einsum('bd,dhk->bhk', x, layer_params['attn']['wk'])
        v = jnp.einsum('bd,dhk->bhk', x, layer_params['attn']['wv'])

        # Update cache at position `step`
        new_ck = cache_k.at[:, step].set(k)
        new_cv = cache_v.at[:, step].set(v)

        # Attention scores: (batch, n_heads, max_seq_len)
        scores = jnp.einsum('bhk,bshk->bhs', q, new_ck) / jnp.sqrt(D_HEAD)
        mask = jnp.arange(MAX_SEQ_LEN) <= step
        scores = jnp.where(mask[None, None, :], scores, -1e9)
        weights = jax.nn.softmax(scores, axis=-1)

        # Weighted sum: (batch, n_heads, d_head)
        attn_out = jnp.einsum('bhs,bshk->bhk', weights, new_cv)
        attn_proj = jnp.einsum('bhk,hkd->bd', attn_out, layer_params['attn']['wo'])

        x = x + attn_proj

        # Feedforward
        h = jnp.maximum(x @ layer_params['ff']['w1'] + layer_params['ff']['b1'], 0)
        ff_out = h @ layer_params['ff']['w2'] + layer_params['ff']['b2']
        x = x + ff_out

        new_caches.append((new_ck, new_cv))

    logits = x @ params['unembed']
    return logits, new_caches

print("Batched decoder step defined.")

Batched decoder step defined.


In [15]:
# @title Sharded Batched Generation { display-mode: "form" }

def make_batched_empty_cache(batch_size):
    return [(jnp.zeros((batch_size, MAX_SEQ_LEN, N_HEADS, D_HEAD)),
             jnp.zeros((batch_size, MAX_SEQ_LEN, N_HEADS, D_HEAD)))
            for _ in range(N_LAYERS)]

@jax.jit
def generate_sharded(params, prompt_tokens, keys):
    """Generate tokens for a batch of prompts.

    Args:
        params: replicated decoder params
        prompt_tokens: (batch, prompt_len) — same prompt for now
        keys: (batch,) PRNG keys, one per batch element
    """
    batch_size = prompt_tokens.shape[0]
    prompt_len = prompt_tokens.shape[1]
    max_new_tokens = 32

    # Prefill: process the prompt
    cache = make_batched_empty_cache(batch_size)
    def prefill_step(carry, token_col):
        cache, step = carry
        _, new_cache = batched_decoder_step(params, token_col, cache, step)
        return (new_cache, step + 1), None

    # token_col: (batch,) at each step — scan over prompt length
    prompt_T = prompt_tokens.T  # (prompt_len, batch) for scanning
    (cache, next_step), _ = lax.scan(prefill_step, (cache, 0), prompt_T)

    # Get logits for first generated token
    logits, cache = batched_decoder_step(
        params, prompt_tokens[:, -1], cache, next_step - 1
    )

    # Decode
    def decode_step(carry, _):
        cache, prev_logits, keys, step = carry
        # Sample (vmap categorical over batch)
        tokens = jax.vmap(lambda k, l: jax.random.categorical(k, l))(
            keys[:, 0], prev_logits
        )
        keys = jax.vmap(lambda k: jax.random.split(k)[0])(keys[:, 0])
        keys = keys[:, None]  # keep shape consistent

        new_logits, new_cache = batched_decoder_step(params, tokens, cache, step)
        return (new_cache, new_logits, keys, step + 1), tokens

    keys_shaped = keys[:, None]  # (batch, 1)
    init_carry = (cache, logits, keys_shaped, next_step)
    _, generated = lax.scan(decode_step, init_carry, jnp.zeros(max_new_tokens))

    return generated.T  # (batch, max_new_tokens)

# Create batched input
batch_size = max(n_devices, 4)  # At least 4 for demonstration
prompt = jnp.broadcast_to(jnp.array([1, 5, 10, 20]), (batch_size, 4))
keys = jax.vmap(jax.random.PRNGKey)(jnp.arange(batch_size))

# Shard the batch across devices
prompt = jax.device_put(prompt, NamedSharding(mesh, P('data', None)))
keys = jax.device_put(keys, NamedSharding(mesh, P('data')))

print(f"Batch size: {batch_size}")
print(f"Prompt sharding: {prompt.sharding.spec}")

# Generate
print("\nCompiling + generating...")
start = time.time()
generated = generate_sharded(sharded_params, prompt, keys)
generated.block_until_ready()
total = time.time() - start

print(f"Generated shape: {generated.shape}  (batch × tokens)")
print(f"Time: {total:.2f}s (includes compilation)")
print(f"\nFirst 3 sequences:")
for i in range(min(3, batch_size)):
    print(f"  Batch {i}: {generated[i]}")

print(f"\nEach batch element used a different PRNG key → different sequences.")
print(f"Batch was sharded across {n_devices} device(s).")

Batch size: 4
Prompt sharding: P('data', None)

Compiling + generating...


Generated shape: (4, 32)  (batch × tokens)
Time: 0.18s (includes compilation)

First 3 sequences:
  Batch 0: [10 33 15 20 39 17 19 26 29  2 37  7 51 46 26 13  5 53 61 16 56 49 20 33
 59 44  8 56 12 48 20 26]
  Batch 1: [36 52  4 30 47 27 32 32 48  7 25 45 32 15 30 20 16 16 14 43 12 16  9 26
 14  5  2 48 63 10 54 12]
  Batch 2: [30 11  2 59 45  2 11  0 15 13 25 12 53 41 11 56  9 63 13 44 20 37 17 53
 62 33 47 61 47 27 56 39]

Each batch element used a different PRNG key → different sequences.
Batch was sharded across 1 device(s).


In [16]:
# @title Benchmark: Compiled Generation { display-mode: "form" }

# Benchmark (exclude compilation)
n_runs = 10
start = time.time()
for i in range(n_runs):
    keys_i = jax.vmap(jax.random.PRNGKey)(jnp.arange(batch_size) + i * batch_size)
    keys_i = jax.device_put(keys_i, NamedSharding(mesh, P('data')))
    generated = generate_sharded(sharded_params, prompt, keys_i)
    generated.block_until_ready()
avg_time = (time.time() - start) / n_runs

tokens_per_second = batch_size * 32 / avg_time
print(f"Average generation time ({batch_size} sequences × 32 tokens):")
print(f"  {avg_time*1000:.1f} ms total")
print(f"  {avg_time/batch_size*1000:.1f} ms per sequence")
print(f"  {tokens_per_second:.0f} tokens/second aggregate")
print(f"\nWith {n_devices} device(s), batch is sharded for data-parallel inference.")

Average generation time (4 sequences × 32 tokens):
  3.4 ms total
  0.8 ms per sequence
  37764 tokens/second aggregate

With 1 device(s), batch is sharded for data-parallel inference.


---

# 10. Summary — What To Do Next

## Key Takeaways

1. **JAX arrays live on devices** — there's no separate distributed mode. You declare shardings, JAX handles communication.

2. **Mesh + PartitionSpec** is the modern sharding API. A mesh names device axes; a partition spec maps array axes to mesh axes.

3. **Data parallelism** (replicate params, shard batch) is the simplest and most common strategy. Gradients are all-reduced automatically.

4. **Tensor parallelism** (shard weight matrices) lets you run models too large for one device. Column-parallel is simplest.

5. **[`jax.debug.visualize_array_sharding`](https://jax.readthedocs.io/en/latest/_autosummary/jax.debug.visualize_array_sharding.html)** is your best friend for debugging. If something is slow, check for unexpected all-gathers.

6. **pmap is legacy** — use [`jax.sharding`](https://jax.readthedocs.io/en/latest/jax.sharding.html) for all new code. `pmap` only parallelized over a single batch axis and had a separate programming model from single-device JAX. The `jax.sharding` API unifies single- and multi-device programming: the same `jit`-compiled function runs correctly on 1 device or 1000.

## What This Notebook Built

We ran **batched, sharded autoregressive generation**:
- Decoder params replicated across devices
- Input batch and KV-cache sharded across the data axis
- Prefill and decode loops implemented with `lax.scan`
- Same code works on 1 device or many

## What's Next

In **Notebook 09: Performance**, we'll:
- Debug and profile this inference pipeline
- Apply buffer donation for the KV-cache
- Implement static-shape bucketing for variable-length inputs
- Add quantization awareness
- AOT-compile for serving

The sharded decoder you built here becomes the optimization target there.